# 03_extract_peak_dhw_within_6mo

This notebook computes thermal stress metrics around recorded coral bleaching events using daily Degree Heating Weeks (DHW) data.

For each bleaching event, a ±182-day time window is analyzed to:

- Identify the maximum DHW value (peak thermal stress)
- Determine the date when this maximum occurs
- Calculate the time difference between bleaching and peak stress

The analysis uses *Daily Global 5km Satellite Coral Bleaching Heat Stress Degree Heating Week*

#### Import libraries

In [10]:
import pandas as pd
import numpy as np
import xarray as xr
import requests
import os
from datetime import datetime, timedelta
from tqdm import tqdm
import gc
import time

#### Import df_exact_dhw database

In [ ]:
# Go up one level from notebooks/ to 01_data_assembly/
root = Path("..")
file_path_import = root / "data" / "intermediate" / "df_exact_dhw.xlsx"
df_exact = pd.read_excel(file_path_import)

##### `ensure_file` FUNCTION

This function ensures that the DHW NetCDF file for a given date is available locally in the `temp_nc/` directory. If not, it downloads the file from NOAA

In [3]:
def ensure_file(date_obj):
    date_str = date_obj.strftime('%Y%m%d')
    year_str = date_obj.strftime('%Y')
    file_name = f"dhw_{date_str}.nc"
    local_path = os.path.join("temp_nc", file_name)
    url = f"https://www.star.nesdis.noaa.gov/pub/socd/mecb/crw/data/5km/v3.1_op/nc/v1.0/daily/dhw/{year_str}/ct5km_dhw_v3.1_{date_str}.nc"

    if not os.path.exists(local_path):
        try:
            r = requests.get(url, timeout=10)
            if r.status_code == 200:
                with open(local_path, "wb") as f:
                    f.write(r.content)
                time.sleep(0.1)
                print(f"Downloaded {file_name}")
            else:
                print(f"Failed to download {file_name} — HTTP {r.status_code}")
                return None
        except Exception as e:
            print(f"Download error for {file_name}: {e}")
            return None
    return local_path

##### `get_valid_dhw` FUNCTION

This function retrieves a valid DHW value when the nearest grid cell contains missing data (`NaN`).

**What it does:**
- Identifies the nearest grid point to the target coordinates
- Searches within a surrounding spatial window (defined by `search_radius`)
- Finds the closest grid cell with a non-missing DHW value
- Returns that value

**Why it's needed**

Due to spatial resolution limitations, satellite-derived datsets may miss DHW data points when grid cells span both terrestrial and marine environments

In [6]:
def get_valid_dhw(ds, lat, lon, search_radius=10):
    try:
        dhw = ds
        lats = ds['lat'].values
        lons = ds['lon'].values

        lat_idx = np.abs(lats - lat).argmin()
        lon_idx = np.abs(lons - lon).argmin()

        lat_start = max(lat_idx - search_radius, 0)
        lat_end = lat_idx + search_radius + 1
        lon_start = max(lon_idx - search_radius, 0)
        lon_end = lon_idx + search_radius + 1

        sub_dhw = ds[lat_start:lat_end, lon_start:lon_end]

        closest_val = None
        min_dist = np.inf

        for i in range(sub_dhw.shape[0]):
            for j in range(sub_dhw.shape[1]):
                val = sub_dhw.values[i, j]
                if not np.isnan(val):
                    grid_lat = lats[lat_start + i]
                    grid_lon = lons[lon_start + j]
                    dist = np.sqrt((lat - grid_lat)**2 + (lon - grid_lon)**2)
                    if dist < min_dist:
                        min_dist = dist
                        closest_val = val

        return float(closest_val) if closest_val is not None else None

    except Exception:
        return None

##### main processing script (serial processing)

For each bleaching event in the dataset:

1. **Define time window**
- A ±182-day window is created around the bleaching date
2. **Retrieve daily DHW data**
- Corresponding NetCDF files are downloaded (if needed)
- DHW values are extracted at the event location
3. **Handle missing values**
- If the nearest grid cell contains NaN, a spatial search is performed
4. **Build time series**
- Daily DHW values are compiled into a time-indexed series
5. **Compute metrics**
- Maximum DHW value within the window
- Date of maximum DHW
- Time difference between bleaching date and peak DHW

This process is repeated for each event in the dataset.
For each bleaching record, daily DHW files are opened one at a time, values are extracted, and files are closed immediately after use. This reduces memory pressure and avoids issues associated with loading many NetCDF files simultaneously.

**Note on parallel processing**

A parallelized version of this workflow was explored to improve runtime. However, the final results presented here were generated using the serial processing workflow for reproducibility and stability.

The parallel implementation attempted to preload all daily NetCDF files using `xarray.open_mfdataset()` and distribute event-level processing across multiple workers. This approach failed during NetCDF loading with a `RuntimeError: NetCDF: HDF error`, likely related to reading many NetCDF files simultaneously and/or backend limitations.

Because this issue was not fully resolved, the parallel workflow is not used in this notebook.

In [8]:
os.makedirs("temp_nc", exist_ok=True)

df_exact['MAX_ANNUAL_DHW'] = np.nan
df_exact['DATE_OF_MAX_DHW'] = pd.NaT
df_exact['DAYS_FROM_MAX_DHW'] = np.nan

for idx, row in tqdm(df_exact.iterrows(), total=len(df_exact), desc="Processing entries"):
    lat = row['LATITUDE']
    lon = row['LONGITUDE']
    bleaching_date = row['DATETIME']

    try:
        start_date = bleaching_date - timedelta(days=182)
        end_date = bleaching_date + timedelta(days=182)
        date_range = pd.date_range(start=start_date, end=end_date, freq='D')

        dhw_values = []
        time_coords = []

        for current_date in date_range:
            file_path = ensure_file(current_date)
            if file_path is None or not os.path.exists(file_path):
                continue
            try:
                ds = xr.open_dataset(file_path, engine="netcdf4")
                dhw_layer = ds['degree_heating_week'].isel(time=0)
                val = dhw_layer.sel(lat=lat, lon=lon, method='nearest').values.item()
                if np.isnan(val):
                    val = get_valid_dhw(dhw_layer, lat, lon, search_radius=10)
                dhw_values.append(val)
                time_coords.append(current_date)
                ds.close()
            except Exception as e:
                print(f" Error reading {file_path} for index {idx}: {e}")
                continue

        dhw_series = pd.Series(dhw_values, index=pd.to_datetime(time_coords)).dropna()
        if dhw_series.empty:
            print(f" All DHW values are NaN for entry at index {idx} (lat: {lat}, lon: {lon}, date: {bleaching_date})")
            continue

        max_val = dhw_series.max()
        max_dates = dhw_series[dhw_series == max_val].index
        first_max_date = min(max_dates)
        offset_days = (bleaching_date - first_max_date).days

        df_exact.at[idx, 'MAX_ANNUAL_DHW'] = max_val
        df_exact.at[idx, 'DATE_OF_MAX_DHW'] = first_max_date
        df_exact.at[idx, 'DAYS_FROM_MAX_DHW'] = offset_days

    except Exception as e:
        print(f" Error at index {idx}: {e}")
        continue

    gc.collect()

Processing entries: 100%|█████████████████████████████████████████████████████| 21008/21008 [69:35:44<00:00, 11.93s/it]

Downloaded dhw_20180701.nc


The following columns are added to the dataset:

- **`MAX_ANNUAL_DHW`**

    The highest DHW value observed within the ±182-day window
- **`DATE_OF_MAX_DHW`**

    The date on which the maximum DHW occurred
- **`DAYS_FROM_MAX_DHW`**

    The number of days between the bleaching event and the peak DHW
  
    - Positive values → bleaching occurred after peak heat stress
    - Negative values → bleaching occurred before peak heat stress
    - Zero → bleaching coincided with peak DHW

##### If `DHW` = 0 AND `MAX_ANNUAL_DHW` = 0, set `DATE_OF_MAX_DHW` to `DATETIME` and `DAYS_FROM_MAX_DHW` to 0.

In [6]:
mask = (df_exact['DHW'] == 0) & (df_exact['MAX_ANNUAL_DHW'] == 0)

num_updated = mask.sum()

df_exact.loc[mask, 'DATE_OF_MAX_DHW'] = df_exact.loc[mask, 'DATETIME']
df_exact.loc[mask, 'DAYS_FROM_MAX_DHW'] = 0

print(f"Updated {num_updated} entries where DHW and MAX_ANNUAL_DHW are both zero.")

Updated 3152 entries where DHW and MAX_ANNUAL_DHW are both zero.


##### Identify the best bleaching report per location within a ±6 month window

**Objective**

Multiple bleaching reports may exist for the same location within a short time period. To improve interpreatibility and downstream analysis, this step identifies the best representative bleaching events closest in time to the peak DHW

**Method**

For each unique location (latitude–longitude pair):

1. Bleaching events are grouped by location 
2. For each event, a ±183-day window is defined around the bleaching date
3. Within this window, all nearby bleaching reports are identified
4. The absolute time difference between each report and the date of maximum DHW is calculated
5. The report closest in time to the peak DHW is flagged as the best representative event
   
**Output**

A new column is created:

- `IS_BEST_REPORT`
   - `True`: the bleaching report closest to peak DHW within its local time window
   - `False`: other reports within the same location/time cluster


In [7]:
df_exact['DATETIME'] = pd.to_datetime(df_exact['DATETIME'])
df_exact['DATE_OF_MAX_DHW'] = pd.to_datetime(df_exact['DATE_OF_MAX_DHW'])

df_exact['IS_BEST_REPORT'] = False

df_exact = df_exact.sort_values(by='DATETIME').reset_index(drop=True)

for (lat, lon), group in df_exact.groupby(['LATITUDE', 'LONGITUDE']):
    for idx, row in group.iterrows():
        date = row['DATETIME']
        max_dhw_date = row['DATE_OF_MAX_DHW']
        
        if pd.isna(max_dhw_date):
            continue
        
        date_min = date - timedelta(days=183)
        date_max = date + timedelta(days=183)

        group_window = group[(group['DATETIME'] >= date_min) & (group['DATETIME'] <= date_max)].copy()
        
        group_window['ABS_DIFF'] = (group_window['DATETIME'] - max_dhw_date).abs()

        if not group_window.empty:
            best_idx = group_window['ABS_DIFF'].idxmin()
            df_exact.loc[best_idx, 'IS_BEST_REPORT'] = True

print(" Best reports flagged based on proximity to DATE_OF_MAX_DHW.")

✅ Best reports flagged based on proximity to DATE_OF_MAX_DHW.


In [8]:
total_rows = len(df_exact)

value_counts = df_exact['IS_BEST_REPORT'].value_counts(dropna=False)

percentages = (value_counts / total_rows) * 100

print("Percentage breakdown of 'IS_BEST_REPORT':")
for val, pct in percentages.items():
    print(f"{val}: {pct:.2f}%")

Percentage breakdown of 'IS_BEST_REPORT':
True: 79.04%
False: 20.96%


##### Exporting expanded database as `df_exact_with_dhw_and_max.xlsx`

In [12]:
file_path_export = root / "data" / "intermediate" / "df_exact_with_dhw_and_max.xlsx"

file_path_export.parent.mkdir(parents=True, exist_ok=True)

df_exact.to_excel(file_path_export, index=False)

print(f"Data exported successfully to {file_path_export}")

Data exported successfully to df_exact_with_dhw_and_max.xlsx
